# A3: Hugging Face `generate` baseline

Before we build our own engine we measure what the standard tool does, on our T4, so every later number has something to beat. Greedy decoding, FP16, prompt and output lengths fixed.

**Definitions** (kept for our own engine in Part E):
- **TTFT** (time to first token) = prefill the prompt + produce one token.
- **Step latency** = `(time for N tokens - TTFT) / (N - 1)`. With a batch each sequence gets one token per step, so this is also the inter-token latency.
- **Decode tokens/s** = `batch / step latency` (all sequences together).
- **MBU** (model bandwidth utilisation) = `steps/s x weight bytes / measured bandwidth`: how much of the memory bandwidth goes to streaming the weights once per step (Lesson A2).

Timings wrap `generate()` with a GPU sync, so they include Python overhead, as a user would feel it. Each point is 5 trials; we report min / median / max.

In [1]:
# Setup: clone or hard-reset the repo to GitHub, cd into it, put it on sys.path, install pytest.
import os, subprocess, sys

REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    # Colab may hold untracked files (e.g. results written by a notebook) that clash with a pull.
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

d8b7874 A3: add baseline notebook



In [2]:
import importlib, json
import torch
import tinyserve.roofline, tinyserve.basics, bench.hf_baseline
for module in (tinyserve.roofline, tinyserve.basics, bench.hf_baseline):
    importlib.reload(module)  # picks up repo edits without restarting the kernel
from tinyserve.basics import load_model
from tinyserve.envinfo import collect_env

DEVICE = "cuda"
print(collect_env())
tokenizer, model = load_model(device=DEVICE, dtype=torch.float16)

A1 = json.load(open("results/A1_shapes.json"))["data"]
A2 = json.load(open("results/A2_gpu.json"))["data"]
WEIGHT_BYTES = A1["weight_bytes"]
MEASURED_BANDWIDTH = A2["measured_bandwidth_gb_s"] * 1e9
SUSTAINED_TFLOPS = A2["measured_fp16_tflops"]
# Matmul parameters only: the embedding table is a lookup, and lm_head runs on one row during generate.
MATMUL_PARAMS = A1["num_params"] - model.get_input_embeddings().weight.numel() - model.get_output_embeddings().weight.numel()
print(f"weights {WEIGHT_BYTES / 1e9:.2f} GB, bandwidth {MEASURED_BANDWIDTH / 1e9:.0f} GB/s, sustained {SUSTAINED_TFLOPS:.1f} TFLOPS, matmul params {MATMUL_PARAMS / 1e6:.0f}M")

{'python': '3.13.16', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'git_sha': 'd8b7874', 'gpu': 'Tesla T4', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'transformers': '5.20.2'}


weights 2.20 GB, bandwidth 241 GB/s, sustained 19.4 TFLOPS, matmul params 969M


## 1. Sanity check
One greedy generation, to be sure the model and `generate` behave before we time anything.

In [3]:
from tinyserve.basics import encode

prompt_ids = encode(tokenizer, "The capital of France is", DEVICE)[None, :]
with torch.no_grad():
    out = model.generate(prompt_ids, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))

The capital of France is Paris.

2. B. The capital of Germany is Berlin.

3. C


## 2. TTFT and prefill (compute-bound)
Prefill pushes the whole prompt through the weights at once, so intensity is about the prompt length: for 128+ tokens we expect to be near the compute roof of Lesson A2 (sustained ~19 TFLOPS). `approx TFLOPS` = `2 x matmul params x prompt tokens / TTFT`; it ignores attention (small here), so it is an estimate.

In [4]:
from bench.hf_baseline import measure_ttft_ms
from tinyserve.roofline import summarize

ttft_rows = []
for prompt_len in (32, 128, 512, 1024):
    stats = summarize(measure_ttft_ms(model, tokenizer, batch=1, prompt_len=prompt_len))
    tflops = 2 * MATMUL_PARAMS * prompt_len / (stats["median"] * 1e-3) / 1e12
    ttft_rows.append({"prompt_len": prompt_len, "ttft_ms": stats, "approx_tflops": tflops})
    print(f"prompt {prompt_len:>5}: TTFT {stats['min']:7.1f} / {stats['median']:7.1f} / {stats['max']:7.1f} ms   ~{tflops:5.1f} TFLOPS")

prompt    32: TTFT    36.5 /    41.5 /    55.1 ms   ~  1.5 TFLOPS


prompt   128: TTFT    37.6 /    38.3 /    41.4 ms   ~  6.5 TFLOPS


prompt   512: TTFT   119.7 /   120.9 /   121.4 ms   ~  8.2 TFLOPS


prompt  1024: TTFT   324.5 /   326.2 /   328.7 ms   ~  6.1 TFLOPS


## 3. Decode throughput vs batch size (memory-bound)
Prompt 128 tokens, 128 new tokens, batch 1 to 32. Compare with the Lesson A2 ceiling `batch x bandwidth / weight bytes`. If HF were perfect, MBU would approach 100% at batch 1. The gap is overhead: Python, many small kernels, KV-cache reads.

In [5]:
from bench.hf_baseline import run_case
from tinyserve.roofline import decode_tokens_per_s_ceiling, model_bandwidth_utilization

PROMPT_LEN, NEW_TOKENS = 128, 128
batch_rows = []
print(f"{'B':>3} {'TTFT ms':>8} {'step ms (min/med/max)':>24} {'decode tok/s':>13} {'ceiling':>8} {'MBU':>6} {'e2e tok/s':>10} {'GB peak':>8}")
for batch in (1, 2, 4, 8, 16, 32):
    row = run_case(model, tokenizer, batch, PROMPT_LEN, NEW_TOKENS)
    steps_per_s = 1000.0 / row["step_ms"]["median"]
    row["mbu"] = model_bandwidth_utilization(steps_per_s, WEIGHT_BYTES, MEASURED_BANDWIDTH)
    row["ceiling_tokens_per_s"] = decode_tokens_per_s_ceiling(WEIGHT_BYTES, MEASURED_BANDWIDTH, batch)
    batch_rows.append(row)
    s = row["step_ms"]
    print(f"{batch:>3} {row['ttft_ms']['median']:8.1f} {s['min']:7.2f}/{s['median']:6.2f}/{s['max']:6.2f}   "
          f"{row['decode_tokens_per_s']['median']:13.0f} {row['ceiling_tokens_per_s']:8.0f} {row['mbu']:6.0%} "
          f"{row['end_to_end_tokens_per_s']['median']:10.0f} {row['peak_memory_gb']:8.2f}")

  B  TTFT ms    step ms (min/med/max)  decode tok/s  ceiling    MBU  e2e tok/s  GB peak


  1     39.7   30.71/ 30.90/ 31.15              32      110    30%         32     2.22


  2     44.3   30.97/ 31.54/ 32.00              63      219    29%         63     2.24


  4     76.6   31.80/ 32.42/ 35.01             123      438    28%        122     2.27


  8    163.3   31.66/ 31.93/ 32.01             251      876    29%        243     2.32


 16    331.2   33.37/ 33.75/ 33.99             474     1753    27%        444     2.44


 32    694.2   52.01/ 52.10/ 52.16             614     3506    18%        560     2.67


## 4. What the KV cache buys (preview of Part C)
Same batch-1 case with `use_cache=False`: every step re-runs the whole sequence. Only 2 trials, because it is slow.

In [6]:
nocache_row = run_case(model, tokenizer, 1, PROMPT_LEN, NEW_TOKENS, trials=2, use_cache=False)
cached_row = batch_rows[0]
print(f"batch 1 step latency: cached {cached_row['step_ms']['median']:.2f} ms, no cache {nocache_row['step_ms']['median']:.2f} ms "
      f"({nocache_row['step_ms']['median'] / cached_row['step_ms']['median']:.1f}x slower)")

batch 1 step latency: cached 30.90 ms, no cache 42.92 ms (1.4x slower)


## 5. Save the baseline and run the tests

In [7]:
from tinyserve.envinfo import save_result

saved_path = save_result("A3_hf_baseline", {
    "config": {"dtype": "float16", "prompt_len": PROMPT_LEN, "new_tokens": NEW_TOKENS, "trials": 5, "decoding": "greedy"},
    "inputs_from": {"weight_bytes": WEIGHT_BYTES, "measured_bandwidth_gb_s": MEASURED_BANDWIDTH / 1e9, "sustained_fp16_tflops": SUSTAINED_TFLOPS},
    "ttft_sweep_batch1": ttft_rows,
    "batch_sweep": batch_rows,
    "batch1_no_cache": nocache_row,
})
print(saved_path)

/content/tinyserve/results/A3_hf_baseline.json


In [8]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_a3_bench.py", "tests/test_a2_roofline.py", "-v"], capture_output=True, text=True)
print(result.stdout[-3500:], result.stderr[-1000:])

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/tinyserve
configfile: pyproject.toml
plugins: typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.5
collecting ... collected 16 items

tests/test_a3_bench.py::test_make_prompt_ids_shape_and_device PASSED     [  6%]
tests/test_a3_bench.py::test_make_prompt_ids_rejects_too_long PASSED     [ 12%]
tests/test_a3_bench.py::test_timed_generate_gives_exact_length PASSED    [ 18%]
tests/test_a3_bench.py::test_run_case_fields PASSED                      [ 25%]
tests/test_a3_bench.py::test_run_case_rejects_single_token PASSED        [ 31%]
tests/test_a3_bench.py::test_measure_ttft_returns_one_value_per_trial PASSED [ 37%]
tests/test_a2_roofline.py::test_mbu_formula PASSED                       [ 43%]
tests/test_a2_roofline.py::test_matmul_flops_and_bytes PASSED            [ 50%]
t

In [9]:
# Print the full results file so it can be copied into the repo from the saved notebook.
print(open(saved_path).read())

{
  "name": "A3_hf_baseline",
  "timestamp_utc": "2026-10-25T06:57:10Z",
  "env": {
    "python": "3.13.16",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
    "git_sha": "d8b7874",
    "gpu": "Tesla T4",
    "torch": "2.11.0+cu130",
    "cuda": "13.0",
    "transformers": "5.20.2"
  },
  "data": {
    "config": {
      "dtype": "float16",
      "prompt_len": 128,
      "new_tokens": 128,
      "trials": 5,
      "decoding": "greedy"
    },
    "inputs_from": {
      "weight_bytes": 2200096768,
      "measured_bandwidth_gb_s": 241.04155712700873,
      "sustained_fp16_tflops": 19.423131009195522
    },
    "ttft_sweep_batch1": [
      {
        "prompt_len": 32,
        "ttft_ms": {
          "min": 36.5014069998324,
          "median": 41.52692299976479,
          "max": 55.05141599996932,
          "n": 5
        },
        "approx_tflops": 1.4933562155893718
      },
      {
        "prompt_len": 128,
        "ttft_ms": {
          "min": 37.63841999989381,
          "media

In [ ]:
# Free the GPU for the next notebook: every kernel on this Colab server keeps its own copy of the model.
import gc
import sys
import torch
for name in ("model", "params"):
    globals().pop(name, None)
for name in ("last_traceback", "last_value", "last_type", "last_exc"):  # an old error can keep tensors alive
    if hasattr(sys, name):
        setattr(sys, name, None)
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")